# Cuaderno v5 · Ejemplos con Gradio y Hub

Todos los ejemplos con `transformers` **v5** (la última), envueltos en **Gradio**
para usarlos sin escribir código, y con el **Hub** funcionando.

> Las demos de CPU funcionan en cualquier entorno; las de GPU necesitan la T4 de Colab.

In [ ]:
%pip install -q transformers gradio huggingface_hub torch
import transformers
print("transformers:", transformers.__version__)

---
## 1 · Análisis de sentimiento (Gradio)

In [ ]:
import gradio as gr
from transformers import pipeline

clasificador = pipeline("sentiment-analysis")

def analizar(texto):
    r = clasificador(texto)[0]
    return f"{r['label']} ({r['score']:.2f})"

gr.Interface(fn=analizar, inputs="text", outputs="text",
             examples=["Me encanta", "Pésimo"]).launch()

---
## 2 · NER (Gradio)

In [ ]:
ner = pipeline("ner", aggregation_strategy="simple")

def entidades(texto):
    r = ner(texto)
    return "\n".join(f"{e['word']} → {e['entity_group']}" for e in r)

gr.Interface(fn=entidades, inputs="text", outputs="text",
             examples=["Apple abrió una tienda en Valencia."]).launch()

---
## 3 · Traducción v5 (AutoModelForSeq2SeqLM + Gradio)

En v5 no existe `pipeline("translation")`; usamos el modelo y el tokenizador directamente.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

modelo = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-es")
tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-es")

def traducir(texto):
    entradas = tokenizer(texto, return_tensors="pt")
    salida = modelo.generate(**entradas)
    return tokenizer.decode(salida[0], skip_special_tokens=True)

gr.Interface(fn=traducir, inputs="text", outputs="text",
             examples=["I like to play basketball"]).launch()

---
## 4 · Resumen v5 (chat model + Gradio)

> ⚠️ Requiere GPU (T4).

In [ ]:
pipe_llm = pipeline("text-generation", model="Qwen/Qwen3-4B-Instruct-2507")

def resumir(texto):
    r = pipe_llm([{"role": "user", "content": f"Resume en 2 frases:\n\n{texto}"}])
    return r[0]["generated_text"][-1]["content"]

gr.Interface(fn=resumir, inputs="text", outputs="text").launch()

---
## 5 · Question answering v5 (chat model + Gradio)

> ⚠️ Requiere GPU (T4). Reutiliza `pipe_llm`.

In [ ]:
def responder(contexto, pregunta):
    r = pipe_llm([{"role": "user", "content":
        f"Contexto: {contexto}\n\nPregunta: {pregunta}\nRespuesta:"}])
    return r[0]["generated_text"][-1]["content"]

gr.Interface(fn=responder,
             inputs=[gr.Textbox(lines=5, label="Contexto"), gr.Textbox(label="Pregunta")],
             outputs="text").launch()

---
## 6 · Describir imagen v5 (VLM + Gradio)

> ⚠️ Requiere GPU (T4).

In [ ]:
pipe_vlm = pipeline("image-text-to-text", model="Qwen/Qwen3-VL-4B-Instruct")

def describir(imagen):
    r = pipe_vlm([{"role": "user", "content": [
        {"type": "image", "image": imagen},
        {"type": "text", "text": "Describe esta imagen."},
    ]}])
    return r[0]["generated_text"][-1]["content"]

gr.Interface(fn=describir, inputs=gr.Image(type="filepath"), outputs="text").launch()

---
## 7 · Hub: login y publicar un modelo

In [ ]:
from huggingface_hub import login, HfApi

login()   # pega tu token de escritura

api = HfApi()
print(api.whoami())   # comprueba tu usuario

# Publicar un modelo (tras afinarlo o reutilizar uno propio)
# from transformers import AutoModelForSequenceClassification
# modelo = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
# modelo.push_to_hub("tu-usuario/mi-modelo")